<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 3 — dbt practical and consolidation

<small>Data Creators › Day 3 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

<a id="how-to-use"></a>

## About this notebook

- Day 3's anchor: the agenda, then one numbered section per topic. Each section runs as a cycle: a demo, an activity or challenge, a shareback, a break.
- Every concept has its diagram; every demo has the cells that run it; every activity has its brief; every challenge points into the training repository.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard lines are the questions the room works through together.
- The last section, **Questions from the room**, collects the day's questions and their answers as they come.

In [ ]:
# Setup: the demo environment on PATH, a sql() helper for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day3-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day3-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a demo warehouse file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 3 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

In [ ]:
# The two demo warehouses the cells below query: local files on the instructor's laptop
PRICE_SHEET = HERE / "demos/price_sheet/warehouse/price_sheet.duckdb"
PORTFOLIO = HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb"

## Today

A ten-minute recap of yesterday's platform, forty minutes of dbt from zero on a small example, then self-paced practice from the README, with the instructor walking the room, all the way to lunch. After lunch, twenty-five minutes of advanced dbt, then more practice. Recap and a pair exercise at the end.

| # | Section | What happens |
|---|---|---|
| 1 | **[dbt from zero: one transformation through the whole life cycle](#s1)** | Recap: yesterday's platform in four pictures · Demo: what dbt is for, why use it, then the build: project, profile, seeds, source, model, Jinja, ref, tests, build, docs |
| 2 | **[The dbt practical: launch, Phase 1 and the walkthrough](#s2)** | Launch: the pipeline · Challenge 9: dbt practical · Walkthrough: Phase 1 · Break · Walkthrough: Phase 2 · Circulating: Phase 2 to lunch · Lunch |
| 3 | **[Advanced dbt, then complete and extend the practical](#s3)** | Demo: a loop, a macro, a custom test, snapshot, incremental, unit test · Challenge 9 continued: Phase 2 and the stretch · Break |
| 4 | **[Consolidation](#s4)** | Activity 9b: second look at yesterday's AI-generated dbt · Debrief: what did we learn? · Activity 9: test review in pairs · Close |

<a id="s1"></a>

## 1. dbt from zero: one transformation through the whole life cycle

### Recap — yesterday's platform in four pictures

Ten minutes to put everyone on the same page. Today's dbt models are SQL that runs on this platform; one question per picture.

<img src="../common/diagrams/png/d3-recap-lakehouse.png" alt="Data warehouse, data lake, lakehouse" style="max-height:400px; max-width:100%;">

**Ask:** What does a lakehouse keep from the lake, and what does it add?

**Answers from the room:**

- 

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Cheap files in object storage, plus a table format (Delta) for transactions and versions, an engine (Spark) for SQL, and a catalog (Unity Catalog) for names and permissions.
- **Today:** Every dbt model becomes a table or a view in that lakehouse.

</details>

![Databricks under the hood: sources, ingestion, Delta storage, Spark compute, Unity Catalog, consumers](../common/diagrams/png/d3-recap-databricks.png)

**Ask:** Where is your data stored, and what runs your SQL?

**Answers from the room:**

- 

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Storage is your S3 bucket in Delta format. Compute is Spark, on a SQL warehouse. Unity Catalog names every table `catalog.schema.table`.
- **Today:** Your tables land in `training_dev.training_<you>`; the raw data arrives with `dbt seed`.

</details>

<img src="../common/diagrams/png/d2-delta-format.png" alt="A Delta table: Parquet files plus a transaction log" style="max-height:400px; max-width:100%;">

**Ask:** What does the log give you that plain Parquet files do not?

**Answers from the room:**

- 

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Every write is a commit: ACID, versions and time travel, MERGE and DELETE on files.
- **Today:** `OPTIMIZE ... ZORDER` in challenge C9 is a Delta operation, run by dbt after each mart is built.

</details>

![How the tools fit together: write, ship, schedule, transform, run, store](../common/diagrams/png/d3-recap-tools.png)

**Ask:** Which tool decides when, which holds the rules, which does the work?

**Answers from the room:**

- 

<details><summary><strong>Facilitator notes</strong></summary>

- **Answer:** Airflow decides when. dbt holds the transformation rules and the tests. Databricks runs the SQL as Spark and writes Delta to S3.
- **Today:** This is the dbt box. Tomorrow: the Airflow box that runs today's project.

</details>

### Demo — one price sheet, from two raw files to a tested, documented table

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday Catalyst generated a dbt project for you. What is in it, and what does dbt do with each file?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>dbt on the smallest useful example: two raw files, four models in three layers, five tests, one documented table, on Day 2's price data.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Every file in today's challenge is one of the kinds shown here. Once you have seen each one written and run, the README's tasks are recognisable.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Watch and predict: each cell on screen writes one file or runs one command. You do not run these cells. In forty minutes you build the real pipeline yourself, on Databricks.</div></div></div>

#### First: what dbt is for

![dbt is the T in extract, load, transform: SQL files in, tables and views out](../common/diagrams/png/d3-what-dbt-does.png)

*dbt is the T in extract, load, transform: SQL files in, tables and views out*

<details><summary>Notes</summary>

- Other tools load the raw data. You write SELECT statements in files, kept in Git; dbt wraps each in a CREATE, runs them in order and tests the results.
- dbt holds no data and does no computing: the warehouse does the work.
- Analysts, analytics engineers and data engineers use the same files. At Capital Group every data product on the paved path has a dbt project; Airflow triggers a Databricks job that runs it.

</details>

#### Why use it: the same price sheet without dbt

In [ ]:
# Without dbt: one SQL script, run by hand. Prices in US dollars = close price x the day's FX rate.
seeds = HERE / "demos/price_sheet/seeds"
sql(f"""
    select p.isin, p.px_date, p.px_ccy, p.px_close,
           round(p.px_close * coalesce(f.rate, 1), 4) as price_usd
    from '{seeds}/vendor_prices_raw.csv' p
    left join '{seeds}/fx_rates_raw.csv' f
      on f.from_ccy = p.px_ccy and f.rate_dt = p.px_date
    where p.px_ccy <> 'USD'
    order by p.isin, p.px_date
""", ":memory:", read_only=False)
# It ran, with no error. Now look at HSBC (GB0005405286): the dollar price equals the pound price. Who would have noticed?

The script ran, and three prices are wrong. The SQL is the easy part; dbt is for everything around it.

<img src="../common/diagrams/png/d3-script-vs-dbt.png" alt="Six questions a hand-run script cannot answer, and what dbt gives you for each" style="max-height:360px; max-width:100%;">

*Six questions a hand-run script cannot answer, and what dbt gives you for each*

#### How a dbt project comes together: the path for the next thirty minutes

![The path: eight steps from an empty folder to a tested, documented table](../common/diagrams/png/d3-journey.png)

*The path: eight steps from an empty folder to a tested, documented table*

<details><summary>Notes</summary>

- Each step adds one kind of file and one command. First make it run (steps 1 to 4), then make it right and say what it is (steps 5 to 8).
- It is the order you would work in on a real product, and the order of the challenge README.

</details>

#### Step 1 — A dbt project is a folder: two YAML files and a folder of SQL

![The anatomy of a dbt project folder](../common/diagrams/png/d3-project-anatomy.png)

*The anatomy of a dbt project folder*

<details><summary>Notes</summary>

- `dbt_project.yml`: the project's name and its defaults. `profiles.yml`: the connection, one entry per target (dev, prod).
- `models/`: one SELECT per `.sql` file, in three folders, plus YAML that describes and tests them. `staging/` cleans one raw table per model (`stg_`), `intermediate/` joins and applies the rules (`int_`), `marts/` holds the tables consumers read (`fct_`, `agg_`). The challenge uses the same folders and prefixes.
- `seeds/`: small CSVs dbt loads. `tests/` and `macros/`: later this morning and after lunch.
- `target/`: what dbt writes when it runs (compiled SQL, run results, the docs site). Never edited, never committed.

</details>

In [ ]:
!bash demos/price_sheet/reset.sh
# The starting point: two raw CSV files and nothing else. `dbt init` would scaffold the folder; here every file is written by hand so you see what is in it.

In [ ]:
%%writefile demos/price_sheet/dbt_project.yml
# dbt_project.yml: what this project is called and how each folder of models is built
name: price_sheet
version: '1.0'
profile: price_sheet          # which connection in profiles.yml to use

models:
  price_sheet:
    staging:                  # models/staging/: one model per raw table, cleaned
      +materialized: view
      +tags: ["staging"]
    intermediate:             # models/intermediate/: the joins and the business rules
      +materialized: view
      +tags: ["intermediate"]
    marts:                    # models/marts/: the tables consumers read
      +materialized: table
      +tags: ["marts"]

In [ ]:
%%writefile demos/price_sheet/profiles.yml
# profiles.yml: where the SQL runs. One project, one or more targets.
price_sheet:
  target: dev
  outputs:
    dev:
      type: duckdb                          # the demo's warehouse: a local file. Yours says: type: databricks
      path: warehouse/price_sheet.duckdb    # ...and yours has: host, http_path, catalog
      schema: demo                          # the schema the models are created in

In [ ]:
!cd demos/price_sheet && dbt debug 2>&1 | grep -E "profiles.yml file|dbt_project.yml file|adapter type|Connection test|All checks"
# dbt debug answers one question: can dbt find the project, find the profile and open the connection? It is the first command in the challenge too.

#### Step 2 — The raw data: seeds stand in for the feeds that land in the warehouse

In [ ]:
!cd demos/price_sheet && dbt seed 2>&1 | grep -E "OK loaded|ERROR|Done"
sql("select isin, px_date, px_close, px_ccy, vendor_ts from demo.vendor_prices_raw where isin in ('US0378331005', 'GB0005405286') order by isin, px_date, vendor_ts", PRICE_SHEET)
# dbt seed loads each CSV in seeds/ as a table. Look closely: HSBC's currency is written gbp, and Apple arrives twice on 4 November. Both matter in a few minutes.

In [ ]:
sql("select * from demo.fx_rates_raw order by rate_dt, from_ccy", PRICE_SHEET)
# One rate per currency per day, to US dollars. The transformation: price in dollars = close price x that day's rate.

#### Step 3 — Declare the source, write the first model, run it

In [ ]:
%%writefile demos/price_sheet/models/sources.yml
# sources.yml: the raw tables this project reads but does not build
version: 2

sources:
  - name: vendor_feed
    description: "End-of-day close prices from the pricing vendor, in each instrument's own currency."
    schema: demo
    tables:
      - name: vendor_prices_raw
  - name: fx_feed
    description: "Daily FX closes to US dollars."
    schema: demo
    tables:
      - name: fx_rates_raw

![YAML files in dbt: .yml and .yaml are the same; only two names are fixed. Today every file is .yml, as in the challenge.](../common/diagrams/png/d3-yaml.png)

*YAML files in dbt: .yml and .yaml are the same; only two names are fixed. Today every file is .yml, as in the challenge.*

<details><summary>Notes</summary>

- `dbt_project.yml` and `profiles.yml` must have exactly those names.
- Every other YAML file can have any name: dbt reads the top-level keys inside (`sources:`, `models:`, `seeds:`, `unit_tests:`), not the file name. `sources.yml` and `schema.yml` are habits; Catalyst writes `_staging__sources.yml` and `_staging__models.yml`.
- You will see `.yaml` in other tools this week: Airflow blueprints, Databricks bundles, data contracts.
- Indentation is the syntax: two spaces per level, never tabs.

</details>

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_prices.sql
-- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    isin                               as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    px_ccy                             as native_ccy,
    cast(vendor_ts as timestamp)       as vendor_ts
from {{ source('vendor_feed', 'vendor_prices_raw') }}

![What dbt run does with a model file: compile, wrap, send](../common/diagrams/png/d3-compile.png)

*What dbt run does with a model file: compile, wrap, send*

<details><summary>Notes</summary>

- Compile: the curly braces are replaced, here `source(...)` becomes the real table name. The result is plain SQL in `target/compiled/`.
- Wrap: dbt puts your SELECT inside `create view ... as` or `create table ... as`, according to the model's materialisation. That is in `target/run/`.
- Send: the warehouse executes it. The model's file name is the name of the view or table.

</details>

In [ ]:
!cd demos/price_sheet && dbt run 2>&1 | grep -E "OK created|ERROR|Done"
sql("select * from demo.stg_prices order by instrument_id, price_date limit 6", PRICE_SHEET)
# One model, one view called stg_prices. You wrote a SELECT; dbt wrote the CREATE.

In [ ]:
print("--- what you wrote: models/staging/stg_prices.sql (last line)")
!tail -1 demos/price_sheet/models/staging/stg_prices.sql
print("\n--- what dbt compiled: target/compiled/ (last line)")
!tail -1 demos/price_sheet/target/compiled/price_sheet/models/staging/stg_prices.sql
print("\n--- what the warehouse ran: target/run/")
!head -6 demos/price_sheet/target/run/price_sheet/models/staging/stg_prices.sql

<img src="../common/diagrams/png/d3-jinja.png" alt="Jinja, the curly braces: dbt replaces them before the warehouse sees the SQL" style="max-height:460px; max-width:100%;">

*Jinja, the curly braces: dbt replaces them before the warehouse sees the SQL*

<details><summary>Notes</summary>

- Two curly braces print a value into the SQL. In the challenge: `source()` from S4, `config()` in S6, `ref()` from C1, `this` in C9.
- A curly brace and a percent sign run logic and print nothing: `if`, `for`, `macro`. Not needed in the challenge; at 13:00.
- Jinja runs on your machine at compile time. When a model misbehaves, read its file under `target/compiled/`.

</details>

#### Step 4 — The other layers with ref(): staging, intermediate, marts, and dbt works out the order

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

In [ ]:
%%writefile demos/price_sheet/models/intermediate/int_prices_usd.sql
-- int_prices_usd: every price joined to its day's FX rate and converted. Same rows as stg_prices.
select
    p.instrument_id,
    p.price_date,
    p.native_ccy,
    p.price_native,
    case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end   as fx_rate_to_usd,
    cast(p.price_native * case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end
         as decimal(18, 6))                                      as price_usd
from {{ ref('stg_prices') }} p
left join {{ ref('stg_fx_rates') }} f
    on  f.from_ccy  = p.native_ccy
    and f.rate_date = p.price_date

In [ ]:
%%writefile demos/price_sheet/models/marts/fct_prices_usd.sql
-- fct_prices_usd: the price sheet consumers read. One row per instrument per day.
select
    instrument_id,
    price_date,
    native_ccy,
    price_native,
    fx_rate_to_usd,
    price_usd
from {{ ref('int_prices_usd') }}

![The graph dbt builds from source() and ref()](../common/diagrams/png/d3-first-dag.png)

*The graph dbt builds from source() and ref()*

<details><summary>Notes</summary>

- Every `source()` and `ref()` is an edge. dbt reads them all, then builds the models from left to right.
- You never write the order down. Add a model and the graph changes by itself.
- The same graph is the lineage picture in the docs, and what `--select +fct_prices_usd` walks.

</details>

In [ ]:
!cd demos/price_sheet && dbt run 2>&1 | grep -E "OK created|ERROR|Done"
# Four models, built in graph order: the two staging models, then the intermediate one, then the mart. Read the log: three say "view model", one says "table model".

In [ ]:
sql("select instrument_id, price_date, native_ccy, price_native, fx_rate_to_usd, price_usd from demo.fct_prices_usd where native_ccy <> 'USD' order by instrument_id, price_date", PRICE_SHEET)
# The run was green. Look at the HSBC rows (GB0005405286): no rate, no dollar price. A successful run says the SQL executed, not that the answer is right.

#### Step 4, continued — View or table? Set per folder in dbt_project.yml, overridden in one model with config()

![A materialisation is how dbt saves a model: view, table, incremental or snapshot](../common/diagrams/png/d3-materialisations.png)

*A materialisation is how dbt saves a model: view, table, incremental or snapshot*

<details><summary>Notes</summary>

- View: a saved query, built on read. Cheap and always fresh. Staging and intermediate, because nobody reads them directly.
- Table: written when dbt runs. Fast to read. Marts, because consumers and dashboards read them.
- Set once per folder in `dbt_project.yml`; `config()` at the top of a model file overrides it for that model. Incremental and snapshot: at 13:00.
- In the challenge: S6, and the `models:` block you read in S2.

</details>

In [ ]:
!awk '/^models:/,0' demos/price_sheet/dbt_project.yml
sql("select table_name, table_type from information_schema.tables where table_schema = 'demo' and table_name not like '%_raw' order by table_name", PRICE_SHEET)
# The folder decides: staging/ and intermediate/ are views, marts/ is a table. Nothing in the SQL files says so.

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
{{ config(materialized='table') }}     -- this one model: a table, whatever the folder says

select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

In [ ]:
!cd demos/price_sheet && dbt run --select stg_fx_rates 2>&1 | grep -E "OK created|ERROR"
sql("select table_name, table_type from information_schema.tables where table_schema = 'demo' and table_name = 'stg_fx_rates'", PRICE_SHEET)
# config() in the file beats the folder setting: the log now says "table model". This is the variation in challenge S6.

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_fx_rates.sql
-- stg_fx_rates: one row per currency per day
select
    from_ccy,
    cast(rate_dt as date)              as rate_date,
    cast(rate as decimal(18, 6))       as fx_rate
from {{ source('fx_feed', 'fx_rates_raw') }}

In [ ]:
!cd demos/price_sheet && dbt run --select stg_fx_rates 2>&1 | grep -E "OK created|ERROR"
# Back to the folder's default, a view: one line removed, nothing else changed.

#### Step 5 — Tests: promises written down, checked on every run

![Two kinds of dbt test, one idea: select the rows that break the rule](../common/diagrams/png/d3-tests.png)

*Two kinds of dbt test, one idea: select the rows that break the rule*

<details><summary>Notes</summary>

- A generic test is a named rule attached to a column in YAML: `not_null`, `unique`, `accepted_values`, `relationships`.
- A singular test is a SQL file in `tests/` for a rule the four cannot express.
- Both compile to a query for the violating rows. No rows returned: pass. The same YAML holds the descriptions the docs show.

</details>

In [ ]:
%%writefile demos/price_sheet/models/staging/schema.yml
# staging/schema.yml: what each staging model and column means, and the promises dbt checks
version: 2

models:
  - name: stg_prices
    description: "Vendor prices, renamed and typed. One row per instrument per day."
    columns:
      - name: instrument_id
        description: "ISIN."
        tests:
          - not_null
      - name: native_ccy
        description: "ISO currency code of the price, upper case."
        tests:
          - accepted_values:
              arguments:
                values: ['USD', 'EUR', 'GBP', 'JPY']

In [ ]:
%%writefile demos/price_sheet/models/marts/schema.yml
# marts/schema.yml: the same, for the table consumers read
version: 2

models:
  - name: fct_prices_usd
    description: "End-of-day prices in US dollars. One row per instrument per day."
    columns:
      - name: instrument_id
        tests:
          - not_null
      - name: price_usd
        description: "Close price times the day's FX rate to USD."
        tests:
          - not_null

In [ ]:
!cd demos/price_sheet && dbt test 2>&1 | grep -E "PASS|FAIL|ERROR|Got|Done"
# Two failures: one currency value that is not in the accepted list (gbp), and three dollar prices that are null. The same three HSBC rows, caught twice.

In [ ]:
!cat demos/price_sheet/target/compiled/price_sheet/models/marts/schema.yml/not_null_fct_prices_usd_price_usd.sql
sql("select native_ccy, count(*) as prices from demo.stg_prices group by native_ccy order by 1", PRICE_SHEET)
# A test is only a query. This one selects the rows where price_usd is null; the cause is two levels upstream, in staging: gbp is not GBP, so the join to the FX rates finds nothing.

#### Step 6 — Fix the model, then dbt build: run and test together, in graph order

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_prices.sql
-- stg_prices: one row per raw price, renamed and typed. No joins, no business logic.
select
    upper(trim(isin))                  as instrument_id,
    cast(px_date as date)              as price_date,
    cast(px_close as decimal(18, 6))   as price_native,
    upper(trim(px_ccy))                as native_ccy,      -- the fix: 'gbp' becomes 'GBP'
    cast(vendor_ts as timestamp)       as vendor_ts
from {{ source('vendor_feed', 'vendor_prices_raw') }}

![The dbt commands, and dbt build that chains them](../common/diagrams/png/d3-commands.png)

*The dbt commands, and dbt build that chains them*

<details><summary>Notes</summary>

- `dbt build` runs seeds, models and tests in graph order; if a test fails, everything downstream of it is skipped.
- `--select` narrows any command: one model, `+model` with everything upstream, `model+` with everything downstream, `tag:staging` for a layer.
- Daily loop: edit a file, `dbt build --select` that model, read the result.

</details>

Every dbt command by objective, the `--select` syntax, the files dbt reads and writes, and what to do when something goes wrong: `common/glossaries/dbt-commands.md`. Keep it open during the challenge.

In [ ]:
!cd demos/price_sheet && dbt build 2>&1 | grep -E "OK|PASS|FAIL|ERROR|SKIP|Done"
sql("select instrument_id, price_date, native_ccy, price_native, fx_rate_to_usd, price_usd from demo.fct_prices_usd where native_ccy = 'GBP' order by price_date", PRICE_SHEET)
# One line changed in staging; build re-created the models and re-ran the four tests. HSBC now has a dollar price.

#### Step 7 — A singular test: any query that returns the rows that break a rule

In [ ]:
%%writefile demos/price_sheet/tests/assert_one_price_per_instrument_per_day.sql
-- A singular test: a query that returns the rows that break the rule. No rows, the test passes.
-- Rule: one price per instrument per day.
select instrument_id, price_date, count(*) as prices
from {{ ref('fct_prices_usd') }}
group by instrument_id, price_date
having count(*) > 1

In [ ]:
!cd demos/price_sheet && dbt test --select assert_one_price_per_instrument_per_day 2>&1 | grep -E "PASS|FAIL|ERROR|Got|Done"
sql("select instrument_id, price_date, price_native, vendor_ts from demo.stg_prices where instrument_id = 'US0378331005' and price_date = '2026-11-04' order by vendor_ts", PRICE_SHEET)
# One failing row: Apple on 4 November has two prices, because the vendor re-delivered a correction. Which one should consumers get? The contract says the latest.

In [ ]:
%%writefile demos/price_sheet/models/staging/stg_prices.sql
-- stg_prices: one row per instrument per day, renamed and typed. No joins, no business logic.
with typed as (
    select
        upper(trim(isin))                  as instrument_id,
        cast(px_date as date)              as price_date,
        cast(px_close as decimal(18, 6))   as price_native,
        upper(trim(px_ccy))                as native_ccy,
        cast(vendor_ts as timestamp)       as vendor_ts
    from {{ source('vendor_feed', 'vendor_prices_raw') }}
),

ranked as (
    select *,
           row_number() over (partition by instrument_id, price_date order by vendor_ts desc) as rn
    from typed
)

-- the fix: when the vendor delivers a price twice, the latest delivery wins
select instrument_id, price_date, price_native, native_ccy, vendor_ts
from ranked
where rn = 1

In [ ]:
!cd demos/price_sheet && dbt build 2>&1 | grep -E "PASS|FAIL|ERROR|SKIP|Done"
sql("select count(*) as price_rows, count(distinct instrument_id) as instruments, min(price_date) as first_day, max(price_date) as last_day from demo.fct_prices_usd", PRICE_SHEET)
# All five tests pass. 36 raw rows became 35 prices: the earlier Apple delivery is gone, by a rule that is written down and tested.

#### Step 8 — Docs and lineage come from the same files

In [ ]:
!cd demos/price_sheet && dbt docs generate 2>&1 | grep -E "Catalog written|Building catalog|ERROR"
!cd demos/price_sheet && dbt -q ls --resource-type model test --select +fct_prices_usd
print("\nTo browse the site: cd demos/price_sheet && dbt docs serve --host 127.0.0.1 --port 8082   (the lineage graph is the blue button, bottom right)")
# The descriptions in schema.yml, the columns from the warehouse and the graph from ref() become a website. Nothing extra was written for it.

![The life cycle of a dbt model, as you just saw it](../common/diagrams/png/d3-lifecycle.png)

*The life cycle of a dbt model, as you just saw it*

<details><summary>Notes</summary>

- Declare the source, write the SELECT, run it, look at the rows.
- Write the promise as a test. When it fails, the fix goes in the model, never in the data.
- Build, document, commit: the pull request carries the SQL, the tests and the descriptions together.

</details>

<img src="../common/diagrams/png/d3-demo-to-challenge.png" alt="From this demo to the challenge: the same kinds of file, more of them" style="max-height:520px; max-width:100%;">

*From this demo to the challenge: the same kinds of file, more of them*

<a id="s2"></a>

## 2. The dbt practical: launch, Phase 1 and the walkthrough

### Launch — the pipeline you will build today

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">You have seen one model through its life cycle. Can you build a pipeline of ten, and tell when it is wrong?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>dbt from the inside: sources, staging, intermediate, marts, tests, docs, on the Portfolio Analytics case, on Databricks.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Reviewing generated code needs the same knowledge as writing it; the practical builds that knowledge one section at a time.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Work the README in order: concept, task, hint, check, variation, solution. Phase 1 by about 11:00, Phase 2 from there and after lunch.</div></div></div>

![The Portfolio Analytics pipeline the learners build: five sources, staging, intermediate, three marts, tests](../common/diagrams/png/d3-pipeline.png)

*The Portfolio Analytics pipeline the learners build: five sources, staging, intermediate, three marts, tests*

<details><summary>Notes</summary>

- Six raw feeds ship as seeds; five are modelled; refdata is there to inspect and for freshness practice.
- Staging renames and casts, one model per source; intermediate does the joins; marts are the consumer-aligned facts and aggregates.
- The tests are promises from a contract made executable: not null, unique, a relationship, accepted values, NAV within one basis point, the rollup reconciles.

</details>

### Challenge 9 — dbt practical: Phase 1 first

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; dbt practical (challenge 02) &nbsp;·&nbsp; about 3 hours across the day</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">Build, test and document the Portfolio Analytics pipeline: eight staging sections, then ten challenges. &nbsp;·&nbsp; <code>challenges/v1/02-dbt-pipeline/README.md</code></div></div>

**Where to work**

1. `cd challenges/v1/02-dbt-pipeline/dbt`; every `dbt` command runs from here.
2. `dbt debug` (connection), `dbt seed` (eight CSVs into `training_dev.training_<you>`), `dbt ls --resource-type model` (ten stubs).
3. `mise run challenge:verify 02` and `challenge:reset 02` run from anywhere; never reset to "start clean": it deletes authored work.

**Phase 1, about an hour**

1. S1 Inspect the raw data.
2. S2 Read the dbt project.
3. S3 Declare sources and freshness.
4. S4 First staging model: `stg_holdings`.
5. S5 The remaining staging models.
6. S6 Materialisations: view vs table.
7. S7 Build the whole staging layer.
8. S8 A staging test.

**Phase 2**

1. C1 The enrichment join: `int_holdings_enriched`.
2. C2 The two supporting intermediate models.
3. C3 First curated mart: `agg_fund_exposure`.
4. C4 Second curated mart: `fct_portfolio_returns`.
5. C5 Generic tests on the marts.
6. C6 A singular reconciliation test: NAV within tolerance.
7. C7 The full build: `dbt build`.
8. C8 Docs and the lineage graph.
9. C9 The Databricks post-hook: `OPTIMIZE … ZORDER`.
10. C10 Capstone: a new mart end to end.

**Gate.** `mise run challenge:verify 02` runs seeds, models and tests; green means solved.

**Support.**

- Keep a list of commands, configuration changes, questions and discoveries; the recap is built from it.

### Walkthrough — Phase 1: sources and staging

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the same project, section by section</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The instructor's copy of the challenge 02 solution in <code>demos/portfolio_analytics/</code>: one cell per README section, for "what should I see?".</div></div>

1. Same model names, layers, tags and tests; the Databricks-only post-hook is a no-op macro here.
2. Use a cell when someone asks what they should see; `demos/portfolio_analytics/README.md` maps each cell to its README section.

#### S1 — Inspect the raw data: what feeds the models

In [ ]:
!cd demos/portfolio_analytics && dbt seed --full-refresh 2>&1 | grep -E "INSERT|Done" 
sql("select * from training_demo.holdings_raw order by fund_cd, pos_dt limit 8", PORTFOLIO)
# A holding: one fund, one instrument, one day, valued in its local currency. Note fund_cd, pos_dt, isin, qty, mkt_val_lc, ccy_cd and _extracted_at.

#### S2 — Read the project: layers, materialisations, tags

In [ ]:
!cd demos/portfolio_analytics && dbt ls --resource-type model --exclude tag:stretch
!sed -n "/^models:/,/^seeds:/p" demos/portfolio_analytics/dbt_project.yml | head -24
# Ten models in three layers. Layers are folders with +materialized and +tags: select a layer with --select tag:staging.

#### S3 — Sources and freshness: five lines per source table, because the fund master has no load timestamp

In [ ]:
!cd demos/portfolio_analytics && dbt source freshness 2>&1 | grep -E "freshness|pass|warn|error|Done" | head -12
# Five sources report; product_ref has no loaded_at_field and is skipped. Tighten error_after to 0 hours and it flips to error: the check working, not a bug.

#### S4 — The first staging model: rename, cast, coalesce a null ISIN into CASH_<ccy>, no joins

In [ ]:
!sed -n "1,30p" demos/portfolio_analytics/models/staging/stg_holdings.sql

In [ ]:
!cd demos/portfolio_analytics && dbt build --select stg_holdings 2>&1 | grep -E "OK|PASS|ERROR|Done"
sql("select fund_id, position_date, instrument_id, quantity, mv_local_ccy, local_ccy from training_demo.stg_holdings order by 1,2 limit 8", PORTFOLIO)
# Same grain as the source. instrument_id is an ISIN or CASH_<ccy>, never null; quantities are typed decimals.

#### S5 to S7 — The remaining staging models, then the whole layer by tag

In [ ]:
!cd demos/portfolio_analytics && dbt build --select tag:staging 2>&1 | grep -E "OK created|PASS|FAIL|ERROR|Done"
sql("select * from training_demo.stg_prices order by instrument_id, price_date limit 8", PORTFOLIO)
# stg_positions and stg_prices dedup to the latest extract per natural key with row_number(); staging never raises the row count.

#### S8 — A staging test is a query that must return no rows

In [ ]:
!sed -n "1,20p" demos/portfolio_analytics/models/staging/schema.yml
!cd demos/portfolio_analytics && dbt test --select stg_holdings 2>&1 | grep -E "PASS|FAIL|Done"
# not_null and unique compile to select-the-violations queries. Zero rows is a pass.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 10:15</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

### Walkthrough — Phase 2: intermediate, marts, tests, docs

#### C1 — The enrichment join: holdings × prices × FX, grain unchanged

In [ ]:
!sed -n "1,45p" demos/portfolio_analytics/models/intermediate/int_holdings_enriched.sql
# Left joins on (instrument, date) and (currency, GBP, date). Ask the room: what happens when the FX rate is missing? Here it defaults to 1.0 via coalesce; the running example on Day 2 chose to leave it null and fail the test. Both are decisions; the contract should say which.

#### C3, C4 — The marts: an aggregate by strategy and the daily-grain fact with a calculated NAV

In [ ]:
!cd demos/portfolio_analytics && dbt build --select +agg_fund_exposure +fct_portfolio_returns 2>&1 | grep -E "OK created|PASS|FAIL|ERROR|Done" | tail -8
sql("select fund_id, position_date, strategy_code, exposure_base_ccy, instrument_count from training_demo.agg_fund_exposure order by 1,2", PORTFOLIO)

In [ ]:
sql("select fund_id, position_date, instrument_id, mv_local_ccy, fx_rate_to_base, mv_base_ccy, round(calc_nav_per_unit, 6) as calc_nav_per_unit from training_demo.fct_portfolio_returns order by 1,2", PORTFOLIO)
# calc_nav_per_unit = total base-currency market value / 100,000,000 synthetic units. Holdings only; cash excluded; a teaching measure, not an official NAV.

#### C5, C6 — Generic tests on the marts, and a singular reconciliation: NAV within one basis point of the custodian

In [ ]:
!sed -n "1,40p" demos/portfolio_analytics/tests/assert_nav_within_tolerance.sql
sql("select c.fund_id, c.position_date, round(c.calc_nav_per_unit,4) as calc, cu.nav_per_unit as custodian, round((c.calc_nav_per_unit - cu.nav_per_unit)/cu.nav_per_unit*10000, 3) as variance_bps from training_demo.fct_portfolio_returns c join training_demo.seed_custodian_nav cu using (fund_id, position_date) order by 1,2", PORTFOLIO)
# The seeds are built so this passes within 1bp. Change one holding value and it will not: that is the test earning its keep.

#### C7, C8 — The full build, then docs and lineage

In [ ]:
!cd demos/portfolio_analytics && dbt build --exclude tag:stretch 2>&1 | grep -E "Done"
!cd demos/portfolio_analytics && dbt docs generate 2>&1 | grep -E "Catalog|Done" | tail -2
print("open target/index.html with: cd demos/portfolio_analytics && dbt docs serve --host 127.0.0.1 --port 8081   (the --host matters: DBT_HOST is used for the Databricks host in the learners' profile)")

#### C9, C10 — The Databricks post-hook and the capstone mart

In [ ]:
!cat demos/portfolio_analytics/macros/optimize_mart.sql
# On Databricks: OPTIMIZE ... ZORDER BY (fund_id, position_date) after each mart, a Delta optimisation. Here: a no-op. A macro that dispatches on target.type is how one project runs on two warehouses.

In [ ]:
sql("select * from training_demo.agg_fund_daily_exposure order by 1,2", PORTFOLIO)
!sed -n "1,20p" demos/portfolio_analytics/tests/reconcile_daily_exposure.sql
# C10: the (fund, date) rollup must preserve the strategy-level total. Learners create both the mart and this test.

### Circulating — Phase 2 to lunch: three questions per learner, and the symptom table

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 CIRCULATION CHECKLIST — three questions per learner</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Ask: could this join pass its tests and still double the reported value? A successful build alone is insufficient.</div></div>

- Sources and staging: can they say what one row means, name its key, and explain the types and units?
- Joins and models: are the lookup keys unique on the right-hand side, do dates belong in the join, and what happens to unmatched rows?
- Tests and output: can they show a value checked independently of the model that produced it?
- Check the row grain, right-side uniqueness, unmatched rows and an independent total.

| Symptom | Cause | Fix |
|---|---|---|
| `auth_type: oauth is required when not using access token` | The shell has no `DBT_ACCESS_TOKEN`; mise did not load the environment | Open a fresh terminal, or `mise run setup:databricks`; do not re-authenticate blindly (the repo's Copilot instructions give a three-step check) |
| `dbt show --inline "select …"` errors with unexpected extra arguments | Running through `mise run challenge:dbt` word-splits the SQL | Run raw `dbt` from the project folder, or use `dbt show --select <model>` |
| `--select tag:staging` selects nothing | The `models:` block in `dbt_project.yml` is missing or mis-keyed | Compare with the shipped `dbt_project.yml`; the project name must match |
| Full build fails in C7 on a placeholder mart | The capstone mart is created in C10, not shipped | Build with `--exclude agg_fund_daily_exposure` until C10 |
| Python crashes on the first query that returns rows | pandas result conversion on this image | The shipped profile sets `_disable_pandas: true`; check it is still there |
| `dbt docs serve` cannot bind | `DBT_HOST` holds the Databricks host | `dbt docs serve --host 127.0.0.1 --port 8081` |
| A relationship test fails after a learner edits a key | Keys changed in staging without the fund master | Trace the key back to the source; staging must not invent keys |

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🍽️ LUNCH &nbsp;·&nbsp; 60 min &nbsp;·&nbsp; back at 13:00</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s3"></a>

## 3. Advanced dbt, then complete and extend the practical

### Demo — Jinja that writes SQL: a loop, a macro, then the production pieces

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">This morning's Jinja filled in names. What else can it do, and what does a production dbt project have that the challenge does not?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A loop and a macro on this morning's price sheet; then a macro that dispatches on the warehouse, a custom generic test, a snapshot (SCD type 2), an incremental mart and a unit test on the Portfolio Analytics project.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>These are the pieces the platform's vended project uses, and what tomorrow's bitemporal macro builds on.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Twenty-five minutes on screen. Then carry on with Phase 2, or ask Copilot to add one of these to your project.</div></div></div>

![A macro: SQL written once, expanded wherever it is called](../common/diagrams/png/d3-macro.png)

*A macro: SQL written once, expanded wherever it is called*

<details><summary>Notes</summary>

- A macro lives in `macros/` and takes arguments, like a function. It returns SQL text.
- Every model that calls it gets the same expression at compile time; change the macro and every model changes.
- The warehouse never sees the macro, only the expanded SQL in `target/compiled/`.

</details>

#### A loop: four columns from one line

In [ ]:
%%writefile demos/price_sheet/models/marts/agg_prices_by_ccy.sql
-- agg_prices_by_ccy: one column per currency, written by a loop instead of by hand
{% set currencies = ['USD', 'EUR', 'GBP', 'JPY'] %}

select
    price_date,
    {% for ccy in currencies %}
    count(case when native_ccy = '{{ ccy }}' then 1 end) as prices_in_{{ ccy | lower }}{% if not loop.last %},{% endif %}
    {% endfor %}
from {{ ref('fct_prices_usd') }}
group by price_date
order by price_date

In [ ]:
!cd demos/price_sheet && dbt show --select agg_prices_by_ccy 2>&1 | tail -8
!sed -n "/^select/,/^from/p" demos/price_sheet/target/compiled/price_sheet/models/marts/agg_prices_by_ccy.sql | grep -v "^ *$"
# The loop ran at compile time and wrote four count(...) lines. Add a currency to the list and the model gains a column.

#### A macro: the conversion rule in one place

In [ ]:
%%writefile demos/price_sheet/macros/to_usd.sql
{# to_usd: the conversion rule, written once. A macro is a function that returns SQL text. #}
{% macro to_usd(amount, ccy, rate) %}
    cast({{ amount }} * case when {{ ccy }} = 'USD' then 1.0 else {{ rate }} end as decimal(18, 6))
{% endmacro %}

In [ ]:
%%writefile demos/price_sheet/models/intermediate/int_prices_usd.sql
-- int_prices_usd: every price joined to its day's FX rate and converted. Same rows as stg_prices.
select
    p.instrument_id,
    p.price_date,
    p.native_ccy,
    p.price_native,
    case when p.native_ccy = 'USD' then 1.0 else f.fx_rate end   as fx_rate_to_usd,
    {{ to_usd('p.price_native', 'p.native_ccy', 'f.fx_rate') }}  as price_usd
from {{ ref('stg_prices') }} p
left join {{ ref('stg_fx_rates') }} f
    on  f.from_ccy  = p.native_ccy
    and f.rate_date = p.price_date

In [ ]:
!cd demos/price_sheet && dbt build --select int_prices_usd+ 2>&1 | grep -E "OK|PASS|FAIL|ERROR|Done"
!sed -n "/fx_rate_to_usd/,/price_usd/p" demos/price_sheet/target/compiled/price_sheet/models/intermediate/int_prices_usd.sql | grep -v "^ *$"
# The model now calls to_usd(...); the compiled SQL shows the expression the macro returned. The tests still pass: same answer, one definition.

#### On the Portfolio Analytics project: a macro that dispatches on the warehouse, and a generic test of your own

In [ ]:
!cat demos/portfolio_analytics/macros/to_base_ccy.sql
!cat demos/portfolio_analytics/tests/generic/test_non_negative.sql
# A macro names a repeated expression once; a generic test is a macro that returns the violating rows.

#### A snapshot: the history of rows that change (SCD type 2)

In [ ]:
!cd demos/portfolio_analytics && dbt snapshot 2>&1 | grep -E "snapshot|Done"
sql("select fund_cd, fund_name, strategy_code, is_active, dbt_valid_from, dbt_valid_to from training_demo.fund_master_snapshot order by 1", PORTFOLIO)
# A snapshot keeps history: change a watched column in the seed, snapshot again, and the old row closes (dbt_valid_to) while a new one opens. Tomorrow adds the second clock.

#### An incremental model: only the new rows

In [ ]:
!cd demos/portfolio_analytics && dbt run --select agg_fund_daily_exposure_incremental 2>&1 | grep -E "incremental|Done"
!cd demos/portfolio_analytics && dbt run --select agg_fund_daily_exposure_incremental 2>&1 | grep -E "incremental|Done"
!sed -n "1,25p" demos/portfolio_analytics/models/stretch/agg_fund_daily_exposure_incremental.sql
# First run: full load. Second run: only dates newer than the table holds, replaced on the unique key (delete+insert) so a re-delivered day does not duplicate.

#### A unit test: the model's logic on made-up rows

In [ ]:
!sed -n "/unit_tests:/,\$p" demos/portfolio_analytics/models/stretch/stretch.yml
!cd demos/portfolio_analytics && dbt test --select stg_holdings,test_type:unit 2>&1 | grep -E "unit_test|PASS|FAIL|Done"
# A unit test runs the model SQL on made-up rows and compares with expected rows: no warehouse data, pure logic.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, in the learner project</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/ai-catalyst adding-dbt-unit-test
Add a dbt unit test for stg_holdings covering: a null ISIN becomes CASH_&lt;ccy&gt;; fund codes are trimmed and upper-cased; a row with a null position date is dropped.</pre><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#C9DAF8;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">INSTRUCTOR</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Claude Code</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">/adding-dbt-unit-test  (same text) — or simply: add a unit_tests entry for stg_holdings in a YAML file covering those three cases</pre></div></div>

### Challenge 9 continued — Phase 2, then your choice of stretch

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#EFEFEF; color:#1a1a1a; padding:8px 14px; font-weight:bold;">☕ BREAK &nbsp;·&nbsp; 15 min &nbsp;·&nbsp; back at 14:20</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Questions go in the table at the end; the next part starts on time.</div></div>

<a id="s4"></a>

## 4. Consolidation

### Activity 9b — second look at yesterday's AI-generated dbt

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 FIRST — two things you could not judge yesterday</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Yesterday afternoon Copilot and Catalyst generated a dbt project for you, and the room asked: do we trust it, review all of it, or review some of it? Write down two things in that project you could not judge.</div></div>

- A file or folder whose purpose you could not tell.
- SQL you could not say was right: a join, a calculation, the curly braces.
- A test you could not say was checking anything.
- Something that looked like more than you asked for.

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 9b &nbsp;—&nbsp; Second look at yesterday's AI-generated dbt &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Yesterday Copilot and Catalyst generated a dbt project from your contract, before you knew dbt. First, write down two things in that project you could not judge yesterday. Then reopen it with a partner (if yours did not finish, use your partner's) and assess it in graph order: the sources, then staging, then the joins, then the tests, then anything nobody asked for. Finally, pick one thing you caught and prompt Copilot again to fix it: name the file, say what is wrong and say how you will check the fix. Read what it changed and run the check.</strong><br><br><strong>Produce:</strong><br>• Two things you could not judge yesterday, each marked: can judge now, or still cannot<br>• One thing you catch now that you would have approved yesterday, with the file it is in<br>• The prompt you wrote to fix it, and whether the fix passed your check<br>• One thing you still cannot judge, for the questions table<br><br><strong>Debrief questions:</strong><br>• What did you catch today that you would have approved yesterday?<br>• How is today's prompt different from yesterday's?<br>• Did the AI write anything nobody asked for? What would you delete?</div></div>

**What to look for, in order**

| Where | Ask |
|---|---|
| `sources.yml` | Is every raw table declared, or are there hard-coded table names in the SQL? |
| Staging models | One row per what? Are names, types and codes cleaned here, and are duplicates handled? |
| Joins | Are the keys unique on the right-hand side? What happens to a row with no match: dropped, null, or a silent default? |
| Tests | Does each promise in your contract (keys, not null, accepted values, a reconciliation) have a test? Would any test fail if the data were wrong? |
| `dbt_project.yml`, `config()` | Which models are views and which are tables, and is that what consumers need? |
| Everything else | Which files, models or macros did nobody ask for? |

Open one model's file under `target/compiled/` if you are unsure what the warehouse receives.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D0E0E3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🤖 PROMPT</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;"><span style="display:inline-block;margin:0 6px 4px 0;border:1px solid #434343;border-radius:4px;overflow:hidden;font-size:0.85em;line-height:1.7;white-space:nowrap;"><span style="background:#D0E0E3;color:#1a1a1a;padding:2px 7px;font-weight:700;letter-spacing:0.05em;">LEARNERS</span><span style="background:#fff;color:#1a1a1a;padding:2px 7px;">Copilot Chat · Chat view, in yesterday's generated project</span></span><pre style="white-space:pre-wrap;background:#f8f8f8;padding:8px;border-radius:4px;">In models/&lt;the file&gt;, &lt;what is wrong: for example, the join to FX rates drops prices with no rate for that date&gt;.
Change only that model: &lt;what you want instead&gt;. Add or tighten a test in the schema YAML that would fail if this went wrong again.
Show me the diff, then run dbt build --select &lt;the model&gt;+ and show me the result.</pre></div></div>

### Debrief — what did we learn?

**🖊️ Whiteboard — what did we learn? (fill gaps from this list, do not lecture it)**

- Models and conventions
- Dependencies and change
- Testing and contracts
- Bitemporal loading
- Deployment context
- Catalyst
- Airflow preview
- Execution and storage
- Catalogue conventions
- Quality and materialisation

### Activity 9 — test review in pairs

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 9 &nbsp;—&nbsp; Test review in pairs &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 10 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Swap one mapping rule and the test that guards it with the person next to you. The reviewer proposes a wrong result the pipeline could produce: a duplicated join, a missing dated FX rate, a changed unit. Decide together whether the test would catch it. Agree one improvement and write down the expected failure.</strong><br><br><strong>Produce:</strong><br>• One rule, its test, and a wrong result the test would or would not catch<br>• One improvement (a new test, a tightened test, or an independent check) and the failure it should produce<br><br><strong>Debrief questions:</strong><br>• Which wrong results would have passed every test in the project?<br>• What value did you check without using the model that produced it?</div></div>

### Close — closing discussion

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ Two things each learner could not judge in yesterday's generated project, marked as judged or still open; one thing they caught, and the prompt that fixed it.<br>☐ The working dbt project, model and test results, one independent output check and the outstanding questions for orchestration, kept where they are.<br>☐ Each learner can trace one plain-English mapping through its model and its test.<br>☐ Two volunteers (or a guest) confirmed for tomorrow's bitemporal and deployment slots, with the brief.<br>☐ The Airflow baseline is ready: Databricks auth works, challenge 02's seeds are loaded, and Astro has been started once after setup.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>